# Reviewer-response dense task runner

**Public-release note.** This notebook is part of a GitHub-ready version of the KMC memristor project. Notebook outputs were stripped to keep the repository lightweight, and path settings were adjusted to use repository-relative locations where needed.

**Purpose:** End-to-end notebook for rerunning the 12 reviewer-response tasks and exporting curated figures and summary tables.

**Main manuscript role:** Used for dense reruns, reviewer-response plots, and auxiliary device-side summary tables.

**Default assumption:** run the notebook from inside this repository so that the `results/` directory can be discovered automatically.



# KMC review-response task runner

This notebook adds a **single end-to-end pipeline cell** that calls the user's earlier KMC notebook functions, reruns the 12 review-driven tasks, saves figures/CSVs, and prints short machine-generated conclusions.

Use pattern:
1. Edit only the paths and knobs in the first code cell.
2. Run the big pipeline cell once.
3. Collect the saved outputs from `OUTPUT_ROOT`.


In [ ]:
from pathlib import Path


def find_repo_root(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for path in [start] + list(start.parents):
        if (path / "README.md").exists() and (path / "notebooks").exists() and (path / "results").exists():
            return path
    return start

REPO_ROOT = find_repo_root()
NOTEBOOK_DIR = REPO_ROOT / "notebooks"

# ============================================================
# 0) Edit only these paths / knobs  --- PAPER-GRADE LARGE RUN
# ============================================================
PHASE2_NOTEBOOK = NOTEBOOK_DIR / "02_phase2_field_temperature_scan.ipynb"
DYNAMIC_NOTEBOOK = NOTEBOOK_DIR / "03_dynamic_feedback_validation.ipynb"
PHASE1_NOTEBOOK = NOTEBOOK_DIR / "01_phase1_order_parameter_scan.ipynb"
SENS_NOTEBOOK = NOTEBOOK_DIR / "04_sensitivity_analysis.ipynb"
SUPP_NOTEBOOK = NOTEBOOK_DIR / "06_supplementary_figures_s1_s8.ipynb"
SCALING_NOTEBOOK = NOTEBOOK_DIR / "05_drive_normalized_scaling.ipynb"

OUTPUT_ROOT = REPO_ROOT / "results" / "review_response"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# ============================================================
# Global run mode
# ============================================================
FAST_SANITY_MODE = False

# Run all tasks by default
TASKS_TO_RUN = list(range(1, 13))

# ============================================================
# Baseline dense sampling grid


In [ ]:

# ============================================================
# 1) End-to-end pipeline cell
# ============================================================
import os, re, json, math, time, shutil, inspect, importlib.util
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display, Markdown
from scipy.optimize import curve_fit
from scipy.stats import spearmanr

sns.set_theme(style="ticks", context="paper")

# ----------------------------
# small utilities
# ----------------------------
def h1(text):
    display(Markdown(f"# {text}"))

def h2(text):
    display(Markdown(f"## {text}"))

def h3(text):
    display(Markdown(f"### {text}"))

def note(text):
    display(Markdown(text))

def ensure_dir(path):
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path

def savefig(fig, outpath, dpi=220):
    outpath = Path(outpath)
    outpath.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(outpath, dpi=dpi, bbox_inches="tight")
    plt.close(fig)

def save_df(df, outpath):
    outpath = Path(outpath)
    outpath.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(outpath, index=False)
    return outpath

def py_literal(v):
    if isinstance(v, Path):
        return repr(str(v))
    return repr(v)

# ----------------------------
# notebook/module loader
# ----------------------------
def read_first_code_cell(nb_path: Path) -> str:
    with open(nb_path, "r", encoding="utf-8") as f:
        nb = json.load(f)
    for cell in nb["cells"]:
        if cell.get("cell_type") == "code":
            src = "".join(cell.get("source", []))
            if src.strip():
                return src
    raise RuntimeError(f"No non-empty code cell found in {nb_path}")

def patch_assignments(src: str, overrides: dict | None = None) -> str:
    if not overrides:
        return src
    out = src
    for key, value in overrides.items():
        pat = rf"(?m)^(\s*{re.escape(key)}\s*=).*$"
        repl = rf"\1 {py_literal(value)}"
        if re.search(pat, out):
            out = re.sub(pat, repl, out)
        else:
            out = f"{key} = {py_literal(value)}\n" + out
    return out

def load_kmc_namespace(nb_path: Path, module_name: str, overrides: dict | None = None, work_dir: Path | None = None):
    src = read_first_code_cell(nb_path)
    src = patch_assignments(src, overrides)
    work_dir = ensure_dir(work_dir or (OUTPUT_ROOT / "_tmp_modules"))
    py_path = work_dir / f"{module_name}.py"
    py_path.write_text(src, encoding="utf-8")
    spec = importlib.util.spec_from_file_location(module_name, py_path)
    mod = importlib.util.module_from_spec(spec)
    import sys
    sys.modules[module_name] = mod
    spec.loader.exec_module(mod)
    return mod

# ----------------------------
# masks / initialization helpers
# ----------------------------
def make_masks(ns, mode: str, seed: int = 0):
    if mode in ["stripe_z", "checkerboard", "stripe_x"]:
        A, B, active, zmax = ns.build_ab_masks(mode=mode, active_zmax_frac=ns.ACTIVE_Z_MAX_FRAC)
        return A, B, active, zmax

    ii, jj = np.meshgrid(np.arange(ns.WIDTH_CELLS), np.arange(ns.THICKNESS_CELLS), indexing="ij")
    zmax = int(max(2, ns.Z_OX_END * ns.ACTIVE_Z_MAX_FRAC))
    active = (jj >= ns.Z_OX_START) & (jj <= zmax)
    rng = np.random.default_rng(seed)

    if mode == "clustered":
        field = rng.normal(size=(ns.WIDTH_CELLS, ns.THICKNESS_CELLS))
        for _ in range(4):
            field = (
                field + np.roll(field, 1, 0) + np.roll(field, -1, 0) +
                np.roll(field, 1, 1) + np.roll(field, -1, 1)
            ) / 5.0
        thr = np.nanmedian(field[active])
        A = active & (field >= thr)
    elif mode == "gradient":
        z_norm = (jj - ns.Z_OX_START) / max((zmax - ns.Z_OX_START), 1)
        A = active & (z_norm <= 0.5)
    else:
        raise ValueError(f"Unknown mask mode: {mode}")

    B = active & (~A)
    if A.sum() == 0 or B.sum() == 0:
        raise RuntimeError(f"Degenerate mask for mode={mode}")
    return A, B, active, zmax

def init_occ_with_masks(ns, m_target: float, seed: int, A_mask, B_mask):
    rng = np.random.default_rng(seed)
    occ = np.zeros((ns.WIDTH_CELLS, ns.THICKNESS_CELLS), dtype=np.int8)
    occ[:, ns.Z_BOT] = ns.S_EBOT
    occ[:, ns.Z_TOP] = ns.S_ETOP

    A_idx = np.argwhere(A_mask)
    B_idx = np.argwhere(B_mask)
    nA, nB = ns.counts_from_m_target(ns.N_INIT_IONS, len(A_idx), len(B_idx), m_target)

    if nA > 0:
        pickA = rng.choice(len(A_idx), size=nA, replace=False)
        for idx in pickA:
            x, z = A_idx[int(idx)]
            occ[x, z] = ns.S_ION
    if nB > 0:
        pickB = rng.choice(len(B_idx), size=nB, replace=False)
        for idx in pickB:
            x, z = B_idx[int(idx)]
            occ[x, z] = ns.S_ION

    if ns.NUCLEATION_MODE == "center":
        occ[ns.WIDTH_CELLS // 2, 1] = ns.S_FIL
    elif ns.NUCLEATION_MODE == "random":
        occ[int(rng.integers(0, ns.WIDTH_CELLS)), 1] = ns.S_FIL
    elif ns.NUCLEATION_MODE == "random_multi":
        xs = rng.choice(np.arange(ns.WIDTH_CELLS), size=min(ns.NUC_SEEDS, ns.WIDTH_CELLS), replace=False)
        for x in xs:
            occ[int(x), 1] = ns.S_FIL
    else:
        xc = (ns.WIDTH_CELLS - 1) / 2.0
        xs = rng.choice(np.arange(ns.WIDTH_CELLS), size=min(ns.NUC_SEEDS, ns.WIDTH_CELLS), replace=False)
        xs = sorted(xs, key=lambda x: abs(x - xc))
        for x in xs:
            occ[int(x), 1] = ns.S_FIL
    return occ

def compute_m_with_masks(ns, occ, A_mask, B_mask):
    ionA = np.count_nonzero((occ == ns.S_ION) & A_mask)
    ionB = np.count_nonzero((occ == ns.S_ION) & B_mask)
    nA = np.count_nonzero(A_mask)
    nB = np.count_nonzero(B_mask)
    cA = ionA / max(nA, 1)
    cB = ionB / max(nB, 1)
    m = (cA - cB) / max(cA + cB, 1e-12)
    return dict(cA=float(cA), cB=float(cB), m_actual=float(m), m_abs=float(abs(m)), ionA=int(ionA), ionB=int(ionB), nA=int(nA), nB=int(nB))

# ----------------------------
# barrier/profile helpers
# ----------------------------
def smooth_profile(x, passes=2):
    y = np.asarray(x, dtype=float).copy()
    for _ in range(passes):
        y2 = y.copy()
        y2[1:-1] = 0.25 * y[:-2] + 0.50 * y[1:-1] + 0.25 * y[2:]
        y = y2
    return y


def make_profile_variant(ns, profile_name: str):
    base = np.asarray(ns.MU_Z_LAYER, dtype=float)
    if profile_name == "baseline":
        mu_z = base.copy()
    elif profile_name == "smooth":
        mu_z = smooth_profile(base, passes=3)
    elif profile_name == "flat":
        mu_z = 0.5 * base + 0.5 * np.full_like(base, base.mean())
    elif profile_name == "surface_biased":
        z = np.linspace(0, 1, len(base))
        mu_z = base + 0.04 * (z - 0.5)
    else:
        raise ValueError(f"Unknown profile_name: {profile_name}")
    mu_z[0] = mu_z[1]
    mu_z[-1] = mu_z[-2]
    mu_x = mu_z + float(ns.MU_X_EXTRA_EV)
    return mu_x, mu_z


def build_barrier_maps_custom(ns, m_abs: float, barrier_seed: int,
                              scheme: str = "baseline",
                              mu_x_layer=None, mu_z_layer=None):
    rng = np.random.default_rng(barrier_seed)
    sigma_ref = float(ns.sigma_from_m(m_abs))
    lc_ref = float(ns.lc_from_m(m_abs))
    sigma_fixed = float(ns.sigma_from_m(0.50))
    lc_fixed = float(ns.lc_from_m(0.50))

    if scheme == "baseline":
        sigma_E = sigma_ref
        lc = lc_ref
    elif scheme == "sigma_only":
        sigma_E = sigma_ref
        lc = lc_fixed
    elif scheme == "lc_only":
        sigma_E = sigma_fixed
        lc = lc_ref
    else:
        raise ValueError(f"Unknown scheme: {scheme}")

    mu_x_layer = np.asarray(ns.MU_X_LAYER if mu_x_layer is None else mu_x_layer, dtype=float)
    mu_z_layer = np.asarray(ns.MU_Z_LAYER if mu_z_layer is None else mu_z_layer, dtype=float)

    eta = ns.correlated_gaussian_field(ns.WIDTH_CELLS, ns.THICKNESS_CELLS, lc, rng)
    eta[:, ns.Z_BOT] = 0.0
    eta[:, ns.Z_TOP] = 0.0

    mu_x_site = np.tile(mu_x_layer, (ns.WIDTH_CELLS, 1))
    mu_z_site = np.tile(mu_z_layer, (ns.WIDTH_CELLS, 1))
    em_x = mu_x_site + sigma_E * eta
    em_z = mu_z_site + sigma_E * eta
    em_x[:, ns.Z_BOT] = ns.BASE_BARRIER_EV
    em_x[:, ns.Z_TOP] = ns.BASE_BARRIER_EV
    em_z[:, ns.Z_BOT] = ns.BASE_BARRIER_EV
    em_z[:, ns.Z_TOP] = ns.BASE_BARRIER_EV
    em_x = np.maximum(em_x, ns.MIN_BARRIER_EV).astype(np.float64)
    em_z = np.maximum(em_z, ns.MIN_BARRIER_EV).astype(np.float64)

    meta = dict(
        sigma_E=float(sigma_E),
        lc=float(lc),
        sigma_ref=float(sigma_ref),
        lc_ref=float(lc_ref),
        eta=eta,
        mu_x_site=mu_x_site,
        mu_z_site=mu_z_site,
        barrier_std_x=float(np.std(em_x[1:-1, 1:-1])),
        barrier_std_z=float(np.std(em_z[1:-1, 1:-1])),
    )
    return em_x, em_z, meta

# ----------------------------
# observables / stats helpers
# ----------------------------
def bottom_top_contact_fraction(ns, occ):
    mask = ns._bfs_bottom_connected_mask(occ)
    bottom = 0.0
    top = 0.0
    if occ.shape[1] >= 3:
        bottom = float(np.mean((mask[:, 1] == 1) & (occ[:, 1] == ns.S_FIL)))
        top = float(np.mean((mask[:, -2] == 1) & (occ[:, -2] == ns.S_FIL)))
    return bottom, top


def conductance_proxy(row):
    neck = max(float(row.get("neck_nm_lrs", np.nan) or np.nan), 1e-9)
    tort = max(float(row.get("tortuosity_nm_lrs", np.nan) or np.nan), 1e-9)
    bcf = max(float(row.get("bottom_contact_frac_lrs", np.nan) or np.nan), 1e-9)
    tcf = max(float(row.get("top_contact_frac_lrs", np.nan) or np.nan), 1e-9)
    return (neck / tort) * np.sqrt(bcf * tcf)


def wilson_interval(k, n, z=1.96):
    if n <= 0:
        return (np.nan, np.nan)
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2*n)) / denom
    half = z * math.sqrt((p*(1-p)/n) + z**2/(4*n*n)) / denom
    return center - half, center + half


def logistic_ab(x, a, b):
    return 1.0 / (1.0 + np.exp(-(a + b*np.asarray(x))))


def fit_logistic_1d(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    keep = np.isfinite(x) & np.isfinite(y)
    x = x[keep]
    y = y[keep]
    if len(np.unique(x)) < 2:
        return dict(ok=False, a=np.nan, b=np.nan, E50=np.nan, wE=np.nan)
    y = np.clip(y, 1e-4, 1-1e-4)
    p0 = [0.0, 20.0]
    try:
        popt, _ = curve_fit(logistic_ab, x, y, p0=p0, maxfev=10000)
        a, b = map(float, popt)
        if abs(b) < 1e-12:
            return dict(ok=False, a=a, b=b, E50=np.nan, wE=np.nan)
        return dict(ok=True, a=a, b=b, E50=-a/b, wE=1.0/abs(b))
    except Exception:
        order = np.argsort(x)
        xs = x[order]
        ys = y[order]
        if ys.min() <= 0.5 <= ys.max():
            E50 = float(np.interp(0.5, ys, xs))
            return dict(ok=False, a=np.nan, b=np.nan, E50=E50, wE=np.nan)
        return dict(ok=False, a=np.nan, b=np.nan, E50=np.nan, wE=np.nan)


def bootstrap_e50(raw_df, m_col="m_target", e_col="E", formed_col="formed", n_boot=400, rng_seed=123):
    rng = np.random.default_rng(rng_seed)
    rows = []
    for m, g in raw_df.groupby(m_col):
        base = g.groupby(e_col)[formed_col].agg(["sum", "count"]).reset_index()
        fit0 = fit_logistic_1d(base[e_col], base["sum"] / base["count"])
        boots = []
        for _ in range(n_boot):
            parts = []
            for E, gg in g.groupby(e_col):
                vals = gg[formed_col].to_numpy(dtype=int)
                samp = rng.choice(vals, size=len(vals), replace=True)
                parts.append((E, samp.mean()))
            bs = pd.DataFrame(parts, columns=[e_col, "p"])
            fitb = fit_logistic_1d(bs[e_col], bs["p"])
            boots.append(fitb["E50"])
        boots = np.asarray([v for v in boots if np.isfinite(v)], dtype=float)
        lo, hi = (np.nan, np.nan) if len(boots) == 0 else np.quantile(boots, [0.025, 0.975])
        rows.append(dict(m_target=float(m), E50=float(fit0["E50"]) if np.isfinite(fit0["E50"]) else np.nan,
                         wE=float(fit0["wE"]) if np.isfinite(fit0["wE"]) else np.nan,
                         E50_lo=float(lo) if np.isfinite(lo) else np.nan,
                         E50_hi=float(hi) if np.isfinite(hi) else np.nan,
                         n_boot=int(len(boots))))
    return pd.DataFrame(rows).sort_values("m_target").reset_index(drop=True)


def bootstrap_median_ci(vals, n_boot=400, rng_seed=123):
    arr = np.asarray(vals, dtype=float)
    arr = arr[np.isfinite(arr)]
    if len(arr) == 0:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(rng_seed)
    med = float(np.median(arr))
    boots = []
    for _ in range(n_boot):
        samp = rng.choice(arr, size=len(arr), replace=True)
        boots.append(np.median(samp))
    lo, hi = np.quantile(boots, [0.025, 0.975])
    return med, float(lo), float(hi)


def summarize_grid(raw_df, n_boot=400):
    rows = []
    for keys, g in raw_df.groupby([c for c in ["task", "scheme", "mask_mode", "profile", "lattice_tag", "m_target", "E", "T"] if c in raw_df.columns], dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)
        cols = [c for c in ["task", "scheme", "mask_mode", "profile", "lattice_tag", "m_target", "E", "T"] if c in raw_df.columns]
        row = dict(zip(cols, keys))
        n = len(g)
        k = int(g["formed"].sum())
        p = k / max(n, 1)
        lo, hi = wilson_interval(k, n)
        row.update(n_runs=n, n_formed=k, p_form=p, p_form_lo=lo, p_form_hi=hi)
        formed = g.loc[g["formed"] == 1]
        for metric in ["t_set", "neck_nm_crit", "tortuosity_nm_crit", "branches_crit", "neck_nm_lrs", "tortuosity_nm_lrs", "branches_lrs", "bottom_contact_frac_lrs", "top_contact_frac_lrs", "G_proxy_lrs", "reset_success"]:
            if metric in g.columns:
                arr = formed[metric].to_numpy(dtype=float) if metric != "reset_success" else g[metric].to_numpy(dtype=float)
                if metric == "t_set":
                    arr = np.log10(arr[np.isfinite(arr) & (arr > 0)])
                    med, mlo, mhi = bootstrap_median_ci(arr, n_boot=n_boot)
                    row["log10_t_set_med"] = med
                    row["log10_t_set_lo"] = mlo
                    row["log10_t_set_hi"] = mhi
                else:
                    med, mlo, mhi = bootstrap_median_ci(arr, n_boot=n_boot)
                    row[f"{metric}_med"] = med
                    row[f"{metric}_lo"] = mlo
                    row[f"{metric}_hi"] = mhi
        rows.append(row)
    return pd.DataFrame(rows).sort_values([c for c in ["task", "scheme", "mask_mode", "profile", "lattice_tag", "m_target", "E", "T"] if c in raw_df.columns]).reset_index(drop=True)


def attach_drive_descriptors(summary_df, e50_df, ns):
    out = summary_df.merge(e50_df[["m_target", "E50", "wE"]], on="m_target", how="left")
    out["phi"] = out["E"] / out["E50"]
    out["Ehat"] = (out["E"] - out["E50"]) / out["wE"]
    out["Delta"] = [ns.delta_from_sigmaT(sig, t) for sig, t in zip(out.get("sigma_E_med", pd.Series(np.nan, index=out.index)), out["T"])] if "sigma_E_med" in out.columns else np.nan
    if "lc_med" in out.columns:
        lc_ref = float(out["lc_med"].dropna().median()) if out["lc_med"].notna().any() else np.nan
        out["lc_over_ref"] = out["lc_med"] / lc_ref
    return out


def attach_row_level_descriptors(raw_df, e50_df, ns):
    out = raw_df.merge(e50_df[["m_target", "E50", "wE"]], on="m_target", how="left")
    out["phi"] = out["E"] / out["E50"]
    out["Ehat"] = (out["E"] - out["E50"]) / out["wE"]
    out["Delta"] = [ns.delta_from_sigmaT(sig, t) for sig, t in zip(out["sigma_E"], out["T"])]
    lc_ref = float(out["lc"].dropna().median()) if out["lc"].notna().any() else np.nan
    out["lc_over_ref"] = out["lc"] / lc_ref
    return out


def binned_curve(df, xcol, ycol, bins=8):
    d = df[[xcol, ycol]].dropna().copy()
    if len(d) < 4:
        return pd.DataFrame(columns=["x", "y"])
    d["bin"] = pd.cut(d[xcol], bins=bins, duplicates="drop")
    out = d.groupby("bin", observed=True).agg(x=(xcol, "mean"), y=(ycol, "mean")).reset_index(drop=True)
    return out


def r2_score_simple(y, yhat):
    y = np.asarray(y, dtype=float)
    yhat = np.asarray(yhat, dtype=float)
    keep = np.isfinite(y) & np.isfinite(yhat)
    y = y[keep]; yhat = yhat[keep]
    if len(y) < 2:
        return np.nan
    ss_res = np.sum((y-yhat)**2)
    ss_tot = np.sum((y-y.mean())**2)
    return np.nan if ss_tot <= 0 else 1 - ss_res/ss_tot


def fit_quadratic_r2(df, xcol, ycol):
    d = df[[xcol, ycol]].dropna().copy()
    if len(d) < 4:
        return np.nan, None
    x = d[xcol].to_numpy(float)
    y = d[ycol].to_numpy(float)
    coef = np.polyfit(x, y, deg=2)
    yhat = np.polyval(coef, x)
    return r2_score_simple(y, yhat), coef


def scan_alpha_for_metric(raw_df, metric_col, alphas, bins=8):
    rows = []
    d = raw_df.loc[raw_df["formed"] == 1].copy()
    if len(d) == 0:
        return pd.DataFrame(columns=["alpha", "R2_binned"])
    lc_ref = float(d["lc"].median()) if d["lc"].notna().any() else 1.0
    for alpha in alphas:
        d["Psi_alpha"] = d["phi"] / (d["Delta"] * np.power(d["lc"] / lc_ref, alpha))
        b = binned_curve(d, "Psi_alpha", metric_col, bins=bins)
        if len(b) >= 4:
            r2, _ = fit_quadratic_r2(b.rename(columns={"x":"Psi_alpha", "y":metric_col}), "Psi_alpha", metric_col)
        else:
            r2 = np.nan
        rows.append(dict(alpha=float(alpha), R2_binned=r2))
    return pd.DataFrame(rows)


def adjacent_significance(raw_df, metric="formed", n_boot=400, rng_seed=123):
    rng = np.random.default_rng(rng_seed)
    ms = sorted(raw_df["m_target"].dropna().unique())
    rows = []
    for m0, m1 in zip(ms[:-1], ms[1:]):
        g0 = raw_df.loc[raw_df["m_target"] == m0, metric].to_numpy(dtype=float)
        g1 = raw_df.loc[raw_df["m_target"] == m1, metric].to_numpy(dtype=float)
        g0 = g0[np.isfinite(g0)]
        g1 = g1[np.isfinite(g1)]
        if len(g0) == 0 or len(g1) == 0:
            rows.append(dict(m_left=m0, m_right=m1, diff=np.nan, lo=np.nan, hi=np.nan, significant=False))
            continue
        diff0 = float(np.mean(g1) - np.mean(g0))
        boots = []
        for _ in range(n_boot):
            s0 = rng.choice(g0, size=len(g0), replace=True)
            s1 = rng.choice(g1, size=len(g1), replace=True)
            boots.append(np.mean(s1) - np.mean(s0))
        lo, hi = np.quantile(boots, [0.025, 0.975])
        rows.append(dict(m_left=m0, m_right=m1, diff=diff0, lo=lo, hi=hi, significant=(lo > 0) or (hi < 0)))
    return pd.DataFrame(rows)

# ----------------------------
# custom simulation runner
# ----------------------------
def simulate_one_custom(ns, m0_target, E_vnm, T, structure_seed=2025, mc_seed=0,
                        scheme="baseline", mask_mode="stripe_z", profile_name="baseline",
                        keep_occ=True):
    A_mask, B_mask, active, zmax = make_masks(ns, mask_mode, seed=structure_seed)
    occ0 = init_occ_with_masks(ns, m0_target, seed=structure_seed, A_mask=A_mask, B_mask=B_mask)
    m_meta = compute_m_with_masks(ns, occ0, A_mask, B_mask)
    mu_x_layer, mu_z_layer = make_profile_variant(ns, profile_name)
    em_x, em_z, meta = build_barrier_maps_custom(ns, m_meta["m_abs"], barrier_seed=structure_seed,
                                                 scheme=scheme, mu_x_layer=mu_x_layer, mu_z_layer=mu_z_layer)

    occ_crit, t_set, connected, rates_s, nsamp, _, _, n_hopx, n_hopz, n_grow, n_diss, n_inj, energy_set = ns.run_set(
        occ0.copy(), em_x, em_z, E_vnm, T, mc_seed
    )
    formed = bool(connected)

    row = dict(
        m_target=float(m0_target), m_actual=float(m_meta["m_actual"]), m_abs=float(m_meta["m_abs"]),
        cA=m_meta["cA"], cB=m_meta["cB"], ionA=m_meta["ionA"], ionB=m_meta["ionB"], nA_sites=m_meta["nA"], nB_sites=m_meta["nB"],
        sigma_E=float(meta["sigma_E"]), lc=float(meta["lc"]),
        barrier_std_x=float(meta["barrier_std_x"]), barrier_std_z=float(meta["barrier_std_z"]),
        Delta=float(ns.delta_from_sigmaT(meta["sigma_E"], T)),
        structure_seed=int(structure_seed), mc_seed=int(mc_seed),
        E=float(E_vnm), T=float(T), formed=int(formed),
        t_set=float(t_set) if formed else np.nan,
        energy_set_ev=float(energy_set), n_hopx=int(n_hopx), n_hopz=int(n_hopz), n_grow=int(n_grow), n_diss=int(n_diss), n_inj=int(n_inj),
        scheme=scheme, mask_mode=mask_mode, profile=profile_name,
        task=np.nan, lattice_tag=getattr(ns, "_LATTICE_TAG", "base")
    )

    if formed:
        mask_crit = ns._bfs_bottom_connected_mask(occ_crit)
        neck_crit, tort_crit, branches_crit, comx_crit, drift_crit, meander_crit = ns._morphology_metrics_filament(occ_crit, mask_crit, ns.CELL_X_NM, ns.DRIFT_THRESH_NM)
        row.update(neck_nm_crit=float(neck_crit), tortuosity_nm_crit=float(tort_crit), branches_crit=int(branches_crit),
                   comx_nm_crit=float(comx_crit), drift_crit=float(drift_crit), meander_rms_nm_crit=float(meander_crit))

        occ_lrs, _, _, _, _, _, _, _, _, _, _, _, _ = ns.run_post_thicken(occ_crit.copy(), em_x, em_z, E_vnm, T, mc_seed + 10000)
        mask_lrs = ns._bfs_bottom_connected_mask(occ_lrs)
        neck_lrs, tort_lrs, branches_lrs, comx_lrs, drift_lrs, meander_lrs = ns._morphology_metrics_filament(occ_lrs, mask_lrs, ns.CELL_X_NM, ns.DRIFT_THRESH_NM)
        bcf_lrs, tcf_lrs = bottom_top_contact_fraction(ns, occ_lrs)
        bcf_crit, tcf_crit = bottom_top_contact_fraction(ns, occ_crit)
        row.update(neck_nm_lrs=float(neck_lrs), tortuosity_nm_lrs=float(tort_lrs), branches_lrs=int(branches_lrs),
                   comx_nm_lrs=float(comx_lrs), drift_lrs=float(drift_lrs), meander_rms_nm_lrs=float(meander_lrs),
                   bottom_contact_frac_crit=float(bcf_crit), top_contact_frac_crit=float(tcf_crit),
                   bottom_contact_frac_lrs=float(bcf_lrs), top_contact_frac_lrs=float(tcf_lrs))
        row["G_proxy_lrs"] = conductance_proxy(row)

        occ_reset, _, connected_reset, *_ = ns.run_reset(occ_lrs.copy(), em_x, em_z, RESET_E_SCALE * E_vnm, T, mc_seed + 20000)
        row["reset_success"] = float(not bool(connected_reset))
    else:
        for metric in ["neck_nm_crit", "tortuosity_nm_crit", "branches_crit", "comx_nm_crit", "drift_crit", "meander_rms_nm_crit",
                       "neck_nm_lrs", "tortuosity_nm_lrs", "branches_lrs", "comx_nm_lrs", "drift_lrs", "meander_rms_nm_lrs",
                       "bottom_contact_frac_crit", "top_contact_frac_crit", "bottom_contact_frac_lrs", "top_contact_frac_lrs", "G_proxy_lrs", "reset_success"]:
            row[metric] = np.nan

    if keep_occ:
        row["occ0"] = occ0
        row["occ_crit"] = occ_crit if formed else None
        row["occ_lrs"] = occ_lrs if formed else None
        row["em_x"] = em_x
        row["em_z"] = em_z
    return row


def run_grid_custom(ns, task_name: str, m_list, E_list, T,
                    n_structure_seeds=3, n_mc_per_structure=2,
                    scheme="baseline", mask_mode="stripe_z", profile_name="baseline",
                    keep_occ=False):
    rows = []
    t0 = time.time()
    n_total = len(m_list) * len(E_list) * n_structure_seeds * n_mc_per_structure
    counter = 0
    for m in m_list:
        for E in E_list:
            for s in range(n_structure_seeds):
                structure_seed = 2025 + s
                for mc in range(n_mc_per_structure):
                    row = simulate_one_custom(ns, m0_target=m, E_vnm=E, T=T,
                                              structure_seed=structure_seed, mc_seed=mc,
                                              scheme=scheme, mask_mode=mask_mode, profile_name=profile_name,
                                              keep_occ=keep_occ)
                    row["task"] = task_name
                    rows.append(row)
                    counter += 1
                    if counter % max(1, n_total // 10) == 0:
                        print(f"[{task_name}] progress: {counter}/{n_total}")
    dt = time.time() - t0
    print(f"[{task_name}] finished {len(rows)} runs in {dt/60:.2f} min")
    return pd.DataFrame(rows)

# ----------------------------
# plotting helpers
# ----------------------------
def plot_e50_curves(e50_df, hue, title, outpath):
    fig, ax = plt.subplots(figsize=(6.0, 4.2))
    for key, g in e50_df.groupby(hue):
        ax.plot(g["m_target"], g["E50"], marker="o", label=str(key))
        if {"E50_lo", "E50_hi"}.issubset(g.columns):
            ax.fill_between(g["m_target"], g["E50_lo"], g["E50_hi"], alpha=0.18)
    ax.set_xlabel("$m_0$")
    ax.set_ylabel("$E_{50}$ (V/nm)")
    ax.set_title(title)
    ax.legend(frameon=False)
    savefig(fig, outpath)


def plot_profile_family(ns, outpath):
    fig, ax = plt.subplots(figsize=(6.0, 4.0))
    z = np.asarray(ns.Z_LAYER_NM)
    for name in TASK4_PROFILES:
        _, mu_z = make_profile_variant(ns, name)
        ax.plot(z, mu_z, marker="o", label=name)
    ax.set_xlabel("Depth z (nm)")
    ax.set_ylabel("$\mu_z(z)$ (eV)")
    ax.set_title("Task 4: barrier-profile variants")
    ax.legend(frameon=False)
    savefig(fig, outpath)


def plot_ci_widths(summary_df, outpath):
    d = summary_df.copy()
    if not {"p_form_lo", "p_form_hi"}.issubset(d.columns):
        return
    d["p_width"] = d["p_form_hi"] - d["p_form_lo"]
    fig, ax = plt.subplots(figsize=(6.2, 4.0))
    ax.hist(d["p_width"].dropna(), bins=15)
    ax.set_xlabel("Wilson CI width for $P_{form}$")
    ax.set_ylabel("Count")
    ax.set_title("Task 6: distribution of formation-probability CI widths")
    savefig(fig, outpath)


def plot_tset_ci(summary_df, outpath):
    d = summary_df.dropna(subset=["log10_t_set_med", "log10_t_set_lo", "log10_t_set_hi"]).copy()
    if len(d) == 0:
        return
    fig, ax = plt.subplots(figsize=(6.0, 4.2))
    for m, g in d.groupby("m_target"):
        g = g.sort_values("E")
        ax.errorbar(g["E"], g["log10_t_set_med"],
                    yerr=[g["log10_t_set_med"] - g["log10_t_set_lo"], g["log10_t_set_hi"] - g["log10_t_set_med"]],
                    marker="o", label=f"m={m:.2f}")
    ax.set_xlabel("E (V/nm)")
    ax.set_ylabel("median log10(t_set / s)")
    ax.set_title("Task 7: switching-time bootstrap intervals")
    ax.legend(frameon=False, fontsize=8)
    savefig(fig, outpath)


def plot_raw_vs_binned(raw_df, metric_col, descriptor_col, outpath, bins=8):
    d = raw_df.loc[raw_df["formed"] == 1].copy()
    fig, ax = plt.subplots(figsize=(6.0, 4.2))
    ax.scatter(d[descriptor_col], d[metric_col], s=16, alpha=0.30, label="raw")
    b = binned_curve(d, descriptor_col, metric_col, bins=bins)
    if len(b) > 0:
        ax.plot(b["x"], b["y"], marker="o", linewidth=2.0, label="binned mean")
    ax.set_xlabel(descriptor_col)
    ax.set_ylabel(metric_col)
    ax.set_title(f"Task 8: raw vs binned — {metric_col} vs {descriptor_col}")
    ax.legend(frameon=False)
    savefig(fig, outpath)


def plot_contact_conductance(summary_df, outdir):
    d = summary_df.copy()
    fig, ax = plt.subplots(figsize=(5.8, 4.0))
    if "bottom_contact_frac_lrs_med" in d.columns:
        for m, g in d.groupby("m_target"):
            ax.plot(g["E"], g["bottom_contact_frac_lrs_med"], marker="o", label=f"m={m:.2f}")
        ax.set_xlabel("E (V/nm)")
        ax.set_ylabel("bottom contact fraction (LRS)")
        ax.set_title("Task 10: LRS bottom-contact fraction")
        ax.legend(frameon=False, fontsize=8)
        savefig(fig, Path(outdir) / "task10_bottom_contact_fraction.png")
    else:
        plt.close(fig)

    if "G_proxy_lrs_med" in d.columns:
        fig, ax = plt.subplots(figsize=(5.8, 4.0))
        for m, g in d.groupby("m_target"):
            ax.plot(g["E"], g["G_proxy_lrs_med"], marker="o", label=f"m={m:.2f}")
        ax.set_xlabel("E (V/nm)")
        ax.set_ylabel("geometry-derived $G_{proxy}$")
        ax.set_title("Task 11: LRS conductance proxy")
        ax.legend(frameon=False, fontsize=8)
        savefig(fig, Path(outdir) / "task11_conductance_proxy.png")

    if "reset_success_med" in d.columns:
        fig, ax = plt.subplots(figsize=(5.8, 4.0))
        for m, g in d.groupby("m_target"):
            ax.plot(g["E"], g["reset_success_med"], marker="o", label=f"m={m:.2f}")
        ax.set_xlabel("E (V/nm)")
        ax.set_ylabel("reset success probability")
        ax.set_title("Task 12: reset probability")
        ax.legend(frameon=False, fontsize=8)
        savefig(fig, Path(outdir) / "task12_reset_probability.png")

# ----------------------------
# main execution
# ----------------------------
h1("Review-response KMC rerun pipeline")
note(
    f"Running tasks: {TASKS_TO_RUN}<br>"
    f"FAST_SANITY_MODE = {FAST_SANITY_MODE}<br>"
    f"Outputs will be saved under: <code>{OUTPUT_ROOT}</code>"
)

base_ns = load_kmc_namespace(PHASE2_NOTEBOOK, module_name="phase2_review_base", work_dir=OUTPUT_ROOT / "_tmp_modules")
base_ns._LATTICE_TAG = "base"

dynamic_ns = None
if Path(DYNAMIC_NOTEBOOK).exists():
    dynamic_ns = load_kmc_namespace(DYNAMIC_NOTEBOOK, module_name="dynamic_review_base", work_dir=OUTPUT_ROOT / "_tmp_modules")

# ----- baseline dataset reused across tasks 5-12 -----
h2("Baseline reference dataset")
baseline_raw = run_grid_custom(
    base_ns, "baseline", BASE_M_LIST, BASE_E_LIST, BASE_T,
    n_structure_seeds=BASE_N_STRUCTURE_SEEDS,
    n_mc_per_structure=BASE_N_MC_PER_STRUCTURE,
    scheme="baseline", mask_mode="stripe_z", profile_name="baseline", keep_occ=True
)
save_df(baseline_raw.drop(columns=[c for c in ["occ0", "occ_crit", "occ_lrs", "em_x", "em_z"] if c in baseline_raw.columns]), OUTPUT_ROOT / "baseline_raw.csv")

baseline_summary = summarize_grid(baseline_raw, n_boot=BOOTSTRAP_N)
save_df(baseline_summary, OUTPUT_ROOT / "baseline_summary.csv")

baseline_e50 = bootstrap_e50(baseline_raw, n_boot=BOOTSTRAP_N)
save_df(baseline_e50, OUTPUT_ROOT / "baseline_e50_bootstrap.csv")
plot_e50_curves(baseline_e50.assign(dataset="baseline"), hue="dataset", title="Baseline $E_{50}(m)$", outpath=OUTPUT_ROOT / "baseline_E50.png")

baseline_raw_desc = attach_row_level_descriptors(baseline_raw, baseline_e50, base_ns)

# ============================================================
# Task 1: sigma/lc decoupling
# ============================================================
results = {}
if 1 in TASKS_TO_RUN:
    h2("Task 1 — sigma/lc decoupling")
    frames = []
    for scheme in ["baseline", "sigma_only", "lc_only"]:
        df = run_grid_custom(
            base_ns, f"task1_{scheme}", TASK1_M_LIST, TASK1_E_LIST, TASK1_T,
            n_structure_seeds=TASK1_N_STRUCTURE_SEEDS,
            n_mc_per_structure=TASK1_N_MC_PER_STRUCTURE,
            scheme=scheme, mask_mode="stripe_z", profile_name="baseline", keep_occ=False
        )
        frames.append(df)
    t1_raw = pd.concat(frames, ignore_index=True)
    t1_summary = summarize_grid(t1_raw, n_boot=BOOTSTRAP_N)
    t1_e50 = pd.concat([
        bootstrap_e50(t1_raw.loc[t1_raw["scheme"] == scheme], n_boot=BOOTSTRAP_N).assign(scheme=scheme)
        for scheme in ["baseline", "sigma_only", "lc_only"]
    ], ignore_index=True)
    save_df(t1_raw, OUTPUT_ROOT / "task1_sigma_lc_raw.csv")
    save_df(t1_summary, OUTPUT_ROOT / "task1_sigma_lc_summary.csv")
    save_df(t1_e50, OUTPUT_ROOT / "task1_sigma_lc_e50.csv")
    plot_e50_curves(t1_e50, hue="scheme", title="Task 1: $E_{50}(m)$ for sigma/lc decoupling", outpath=OUTPUT_ROOT / "task1_sigma_lc_E50.png")

    pivot = t1_e50.pivot(index="m_target", columns="scheme", values="E50")
    rmse_sigma = float(np.sqrt(np.nanmean((pivot["baseline"] - pivot["sigma_only"])**2))) if {"baseline", "sigma_only"}.issubset(pivot.columns) else np.nan
    rmse_lc = float(np.sqrt(np.nanmean((pivot["baseline"] - pivot["lc_only"])**2))) if {"baseline", "lc_only"}.issubset(pivot.columns) else np.nan
    if np.isfinite(rmse_sigma) and np.isfinite(rmse_lc):
        if rmse_sigma < rmse_lc:
            txt = f"sigma-only reproduces baseline $E_{{50}}(m)$ better (RMSE={rmse_sigma:.4f}) than lc-only (RMSE={rmse_lc:.4f}), so the threshold trend is more strongly controlled by $\\sigma_E$ than by $l_c$."
        else:
            txt = f"lc-only is at least as influential as sigma-only for the threshold trend (RMSE sigma-only={rmse_sigma:.4f}, lc-only={rmse_lc:.4f})."
    else:
        txt = "Task 1 finished, but RMSE comparison could not be evaluated because one or more E50 fits were not finite."
    note("**Task 1 conclusion.** " + txt)
    results[1] = dict(raw=t1_raw, summary=t1_summary, e50=t1_e50)

# ============================================================
# Task 2: mask robustness
# ============================================================
if 2 in TASKS_TO_RUN:
    h2("Task 2 — order-mask robustness")
    frames = []
    for mask_mode in TASK2_MASKS:
        df = run_grid_custom(
            base_ns, f"task2_{mask_mode}", TASK2_M_LIST, TASK2_E_LIST, TASK2_T,
            n_structure_seeds=TASK2_N_STRUCTURE_SEEDS,
            n_mc_per_structure=TASK2_N_MC_PER_STRUCTURE,
            scheme="baseline", mask_mode=mask_mode, profile_name="baseline", keep_occ=False
        )
        frames.append(df)
    t2_raw = pd.concat(frames, ignore_index=True)
    t2_summary = summarize_grid(t2_raw, n_boot=BOOTSTRAP_N)
    t2_e50 = pd.concat([bootstrap_e50(t2_raw.loc[t2_raw["mask_mode"] == m], n_boot=BOOTSTRAP_N).assign(mask_mode=m) for m in TASK2_MASKS], ignore_index=True)
    save_df(t2_raw, OUTPUT_ROOT / "task2_mask_raw.csv")
    save_df(t2_summary, OUTPUT_ROOT / "task2_mask_summary.csv")
    save_df(t2_e50, OUTPUT_ROOT / "task2_mask_e50.csv")
    plot_e50_curves(t2_e50, hue="mask_mode", title="Task 2: mask robustness of $E_{50}(m)$", outpath=OUTPUT_ROOT / "task2_mask_E50.png")

    slope_rows = []
    for mask_mode, g in t2_e50.groupby("mask_mode"):
        g = g.dropna(subset=["m_target", "E50"])
        if len(g) >= 2:
            coef = np.polyfit(g["m_target"], g["E50"], 1)
            slope_rows.append((mask_mode, coef[0]))
    slope_df = pd.DataFrame(slope_rows, columns=["mask_mode", "slope"]) if slope_rows else pd.DataFrame(columns=["mask_mode", "slope"])
    save_df(slope_df, OUTPUT_ROOT / "task2_mask_slopes.csv")
    n_same_sign = int((np.sign(slope_df["slope"]) == np.sign(slope_df["slope"].median())).sum()) if len(slope_df) else 0
    note(f"**Task 2 conclusion.** {n_same_sign}/{max(len(slope_df),1)} mask variants preserve the same sign of the $E_{{50}}$–$m$ trend, which is the main robustness check requested for the order-mask definition.")
    results[2] = dict(raw=t2_raw, summary=t2_summary, e50=t2_e50)

# ============================================================
# Task 3: lattice-size / resolution robustness
# ============================================================
if 3 in TASKS_TO_RUN:
    h2("Task 3 — lattice-size / resolution robustness")
    frames = []
    e50_parts = []
    for cfg in TASK3_LATTICES:
        overrides = {
            "WIDTH_CELLS": cfg["width"],
            "THICKNESS_CELLS": cfg["thickness"],
            "XSPAN_NM": cfg["xspan_nm"],
            "TOX_NM": cfg["tox_nm"],
        }
        ns_lat = load_kmc_namespace(PHASE2_NOTEBOOK, module_name=f"phase2_lat_{cfg['tag']}", overrides=overrides, work_dir=OUTPUT_ROOT / "_tmp_modules")
        ns_lat._LATTICE_TAG = cfg["tag"]
        df = run_grid_custom(
            ns_lat, f"task3_{cfg['tag']}", TASK3_M_LIST, TASK3_E_LIST, TASK3_T,
            n_structure_seeds=TASK3_N_STRUCTURE_SEEDS,
            n_mc_per_structure=TASK3_N_MC_PER_STRUCTURE,
            scheme="baseline", mask_mode="stripe_z", profile_name="baseline", keep_occ=False
        )
        frames.append(df)
        e50_parts.append(bootstrap_e50(df, n_boot=BOOTSTRAP_N).assign(lattice_tag=cfg["tag"]))
    t3_raw = pd.concat(frames, ignore_index=True)
    t3_summary = summarize_grid(t3_raw, n_boot=BOOTSTRAP_N)
    t3_e50 = pd.concat(e50_parts, ignore_index=True)
    save_df(t3_raw, OUTPUT_ROOT / "task3_lattice_raw.csv")
    save_df(t3_summary, OUTPUT_ROOT / "task3_lattice_summary.csv")
    save_df(t3_e50, OUTPUT_ROOT / "task3_lattice_e50.csv")
    plot_e50_curves(t3_e50, hue="lattice_tag", title="Task 3: lattice-size robustness of $E_{50}(m)$", outpath=OUTPUT_ROOT / "task3_lattice_E50.png")

    spread = t3_e50.groupby("m_target")["E50"].agg(lambda s: np.nanmax(s)-np.nanmin(s)).reset_index(name="E50_spread")
    save_df(spread, OUTPUT_ROOT / "task3_lattice_spread.csv")
    note(f"**Task 3 conclusion.** Median cross-lattice spread in fitted $E_{{50}}$ is {np.nanmedian(spread['E50_spread']):.4f} V/nm. Use this number directly in the rebuttal to quantify discretization sensitivity.")
    results[3] = dict(raw=t3_raw, summary=t3_summary, e50=t3_e50)

# ============================================================
# Task 4: barrier-profile robustness
# ============================================================
if 4 in TASKS_TO_RUN:
    h2("Task 4 — barrier-profile robustness")
    plot_profile_family(base_ns, OUTPUT_ROOT / "task4_profile_family.png")
    frames = []
    e50_parts = []
    for profile_name in TASK4_PROFILES:
        df = run_grid_custom(
            base_ns, f"task4_{profile_name}", TASK4_M_LIST, TASK4_E_LIST, TASK4_T,
            n_structure_seeds=TASK4_N_STRUCTURE_SEEDS,
            n_mc_per_structure=TASK4_N_MC_PER_STRUCTURE,
            scheme="baseline", mask_mode="stripe_z", profile_name=profile_name, keep_occ=False
        )
        frames.append(df)
        e50_parts.append(bootstrap_e50(df, n_boot=BOOTSTRAP_N).assign(profile=profile_name))
    t4_raw = pd.concat(frames, ignore_index=True)
    t4_summary = summarize_grid(t4_raw, n_boot=BOOTSTRAP_N)
    t4_e50 = pd.concat(e50_parts, ignore_index=True)
    save_df(t4_raw, OUTPUT_ROOT / "task4_profile_raw.csv")
    save_df(t4_summary, OUTPUT_ROOT / "task4_profile_summary.csv")
    save_df(t4_e50, OUTPUT_ROOT / "task4_profile_e50.csv")
    plot_e50_curves(t4_e50, hue="profile", title="Task 4: profile robustness of $E_{50}(m)$", outpath=OUTPUT_ROOT / "task4_profile_E50.png")

    rank_rows = []
    for profile, g in t4_e50.groupby("profile"):
        rr, _ = spearmanr(g["m_target"], g["E50"], nan_policy="omit")
        rank_rows.append(dict(profile=profile, spearman_r=rr))
    rank_df = pd.DataFrame(rank_rows)
    save_df(rank_df, OUTPUT_ROOT / "task4_profile_rankcorr.csv")
    note("**Task 4 conclusion.** Barrier-profile variants were rerun explicitly. The saved Spearman table gives the sign and strength of the $E_{50}$–$m$ trend for each plausible depth-profile family.")
    results[4] = dict(raw=t4_raw, summary=t4_summary, e50=t4_e50)

# ============================================================
# Task 5: E50 bootstrap confidence intervals
# ============================================================
if 5 in TASKS_TO_RUN:
    h2("Task 5 — bootstrap confidence intervals for $E_{50}$")
    width = baseline_e50.assign(E50_width=lambda d: d["E50_hi"] - d["E50_lo"])
    save_df(width, OUTPUT_ROOT / "task5_e50_bootstrap_widths.csv")
    fig, ax = plt.subplots(figsize=(6.0, 4.2))
    ax.errorbar(width["m_target"], width["E50"],
                yerr=[width["E50"] - width["E50_lo"], width["E50_hi"] - width["E50"]],
                marker="o")
    ax.set_xlabel("$m_0$")
    ax.set_ylabel("$E_{50}$ (V/nm)")
    ax.set_title("Task 5: bootstrap CI for $E_{50}$")
    savefig(fig, OUTPUT_ROOT / "task5_e50_bootstrap_ci.png")
    note(f"**Task 5 conclusion.** Median 95% CI width of $E_{{50}}$ is {np.nanmedian(width['E50_width']):.4f} V/nm for the baseline dataset.")

# ============================================================
# Task 6: Wilson/binomial confidence intervals for Pform
# ============================================================
if 6 in TASKS_TO_RUN:
    h2("Task 6 — Wilson confidence intervals for $P_{form}$")
    save_df(baseline_summary, OUTPUT_ROOT / "task6_baseline_summary_with_pform_ci.csv")
    plot_ci_widths(baseline_summary, OUTPUT_ROOT / "task6_pform_ci_width_hist.png")
    mean_width = float(np.nanmean(baseline_summary["p_form_hi"] - baseline_summary["p_form_lo"]))
    note(f"**Task 6 conclusion.** Mean Wilson CI width for $P_{{form}}$ is {mean_width:.4f}. This directly answers the review request for error bars on formation statistics.")

# ============================================================
# Task 7: bootstrap intervals for t_set
# ============================================================
if 7 in TASKS_TO_RUN:
    h2("Task 7 — bootstrap intervals for switching time")
    plot_tset_ci(baseline_summary, OUTPUT_ROOT / "task7_tset_bootstrap_ci.png")
    d = baseline_summary.dropna(subset=["log10_t_set_lo", "log10_t_set_hi"]).copy()
    width = d["log10_t_set_hi"] - d["log10_t_set_lo"]
    note(f"**Task 7 conclusion.** Median 95% bootstrap interval width for median log10(t_set) is {np.nanmedian(width):.4f} decades.")

# ============================================================
# Task 8: raw vs binned collapse quality
# ============================================================
if 8 in TASKS_TO_RUN:
    h2("Task 8 — raw vs binned collapse quality")
    raw = baseline_raw_desc.loc[baseline_raw_desc["formed"] == 1].copy()
    alpha_scan_df = scan_alpha_for_metric(raw, metric_col="branches_crit", alphas=ALPHA_SCAN, bins=COLLAPSE_BINS)
    save_df(alpha_scan_df, OUTPUT_ROOT / "task8_alpha_scan_branches.csv")
    best_alpha = alpha_scan_df.sort_values("R2_binned", ascending=False).iloc[0]["alpha"] if len(alpha_scan_df) else 0.0
    lc_ref = float(raw["lc"].median()) if raw["lc"].notna().any() else 1.0
    raw["Psi_alpha"] = raw["phi"] / (raw["Delta"] * np.power(raw["lc"] / lc_ref, best_alpha))
    plot_raw_vs_binned(raw, metric_col="branches_crit", descriptor_col="Psi_alpha", outpath=OUTPUT_ROOT / "task8_raw_vs_binned_branches.png", bins=COLLAPSE_BINS)
    b = binned_curve(raw, "Psi_alpha", "branches_crit", bins=COLLAPSE_BINS)
    r2_raw, _ = fit_quadratic_r2(raw, "Psi_alpha", "branches_crit")
    r2_bin, _ = fit_quadratic_r2(b.rename(columns={"x":"Psi_alpha", "y":"branches_crit"}), "Psi_alpha", "branches_crit")
    note(f"**Task 8 conclusion.** Best scanned alpha is {best_alpha:.3f}. Raw-data quadratic $R^2$ = {r2_raw:.3f}; binned-data quadratic $R^2$ = {r2_bin:.3f}. Report both values to avoid overstating collapse quality.")

# ============================================================
# Task 9: intermediate-m significance
# ============================================================
if 9 in TASKS_TO_RUN:
    h2("Task 9 — significance of intermediate-m deviations")
    sig_df = adjacent_significance(baseline_raw, metric="formed", n_boot=BOOTSTRAP_N)
    save_df(sig_df, OUTPUT_ROOT / "task9_adjacent_significance_pform.csv")
    n_sig = int(sig_df["significant"].sum()) if "significant" in sig_df.columns else 0
    note(f"**Task 9 conclusion.** {n_sig}/{max(len(sig_df),1)} adjacent m-intervals show a non-overlapping 95% bootstrap difference in mean formation probability. Non-significant intermediate deviations should not be overinterpreted.")

# ============================================================
# Tasks 10-12: contact fraction, conductance proxy, reset probability
# ============================================================
if any(t in TASKS_TO_RUN for t in [10, 11, 12]):
    h2("Tasks 10-12 — optional device-side observables")
    plot_contact_conductance(baseline_summary, OUTPUT_ROOT)
    cols = [c for c in ["bottom_contact_frac_lrs_med", "G_proxy_lrs_med", "reset_success_med"] if c in baseline_summary.columns]
    if cols:
        save_df(baseline_summary[[c for c in ["m_target", "E"] + cols if c in baseline_summary.columns]], OUTPUT_ROOT / "task10_12_device_side_summary.csv")
    if 10 in TASKS_TO_RUN and "bottom_contact_frac_lrs_med" in baseline_summary.columns:
        corr10 = spearmanr(baseline_summary["m_target"], baseline_summary["bottom_contact_frac_lrs_med"], nan_policy="omit").statistic
        note(f"**Task 10 conclusion.** Spearman correlation between m and LRS bottom-contact fraction is {corr10:.3f} for the pooled baseline summary.")
    if 11 in TASKS_TO_RUN and "G_proxy_lrs_med" in baseline_summary.columns:
        corr11 = spearmanr(baseline_summary["m_target"], baseline_summary["G_proxy_lrs_med"], nan_policy="omit").statistic
        note(f"**Task 11 conclusion.** Spearman correlation between m and the geometry-derived conductance proxy is {corr11:.3f}. Keep the word 'proxy' in the manuscript unless you later add an explicit transport model.")
    if 12 in TASKS_TO_RUN and "reset_success_med" in baseline_summary.columns:
        corr12 = spearmanr(baseline_summary["m_target"], baseline_summary["reset_success_med"], nan_policy="omit").statistic
        note(f"**Task 12 conclusion.** Spearman correlation between m and reset probability is {corr12:.3f} for the chosen reset protocol (using the current code's LRS-to-reset routine).")

h2("Done")
note(f"All requested outputs have been written to <code>{OUTPUT_ROOT}</code>.")
